<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/19_synthetic_data_and_augmentation_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Synthetic Data and Augmentation with LLMs

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Coding texts by hand is the expensive part of supervised learning. A tempting shortcut: let an LLM
write extra training examples. It can **reword** the examples you have (augmentation), or write
**new** examples from scratch (synthetic data). Does a classifier trained on them get better, and
how would you know whether the synthetic texts are any good?

We start from a deliberately hard situation: only **10 coded bill titles per policy area**. We then
compare a classifier trained on those 40 titles with classifiers trained on LLM paraphrases,
on synthetic titles, and on a mix, and check the synthetic data for diversity, realism, label
quality and leakage. As always, the test set is real, human-coded data.

> ⚠️ **Turn on the GPU first:** *Runtime → Change runtime type → T4 GPU.*

In this notebook you will:

- know what synthetic data can and can't do in social science, and how it differs from synthetic respondents
- augment a small labeled set with LLM paraphrases
- generate synthetic training data from category definitions, with and without real examples
- compare classifiers trained on real, augmented and synthetic data on the same real test set
- check synthetic data for diversity, realism, label quality and leakage

**Contents**

1. What synthetic data is for
2. The data: only 40 coded titles
3. Baselines: few labels and many
4. The generator
5. Augmentation: rewording real examples
6. Synthetic data from scratch
7. Did it help?
8. Is the synthetic data any good?
9. Risks
10. Wrap-up

There are 5 exercises along the way, marked ✏️. The solutions notebook has worked answers, but try them yourself first.

## 0. Setup

In [ ]:
!pip install -q transformers accelerate

import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

SEED = 42
set_seed(SEED)
rng = np.random.default_rng(SEED)
pd.set_option("display.max_colwidth", 130)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime → Change runtime type → T4 GPU")

---
## 1. What synthetic data is for

| Use | What the LLM writes | Typical goal |
|---|---|---|
| **Augmentation** | rewordings of your coded examples | more variety from the same labels |
| **Synthetic training data** | new examples for each category | labels where you have few or none, e.g. a rare category |
| **Test cases** | targeted variations (Bonus 16, 17) | probe what a classifier gets wrong |
| **Shareable stand-ins** | data that resembles confidential data | share or teach without the originals |

What it is **not**: a replacement for real people or real texts as the object of study. "Silicon
samples" that stand in for survey respondents are a separate, contested idea (Argyle et al., 2023;
Bisbee et al., 2024). Here the synthetic texts only train a classifier; everything is evaluated on
real, human-coded data.

---
## 2. The data: only 40 coded titles

The bill titles and the test set from Day 4 (25 titles per area). From the remaining pool we draw
the **seed set**: the 10 titles per area we pretend to have coded. We also draw 200 more per area,
to see what coding more real data would have bought.

In [ ]:
parts = []
for chunk in [0, 5]:          # two of the ten Parquet files, as on Day 3
    path = hf_hub_download(repo_id="dreamproit/bill_labels_us", repo_type="dataset",
                           revision="refs/convert/parquet", filename=f"default/train/{chunk:04d}.parquet")
    parts.append(pd.read_parquet(path, columns=["title", "policy_area"]))
raw = pd.concat(parts, ignore_index=True)

LABELS = ["Health", "Education", "Taxation", "Armed Forces and National Security"]
bills = (raw.assign(title=raw["title"].fillna("").str.strip())
            .query("title != ''")
            .drop_duplicates(subset="title"))
bills = bills[bills["policy_area"].isin(LABELS)].sample(frac=1, random_state=SEED).reset_index(drop=True)

dev = bills.groupby("policy_area").head(10)
rest = bills.drop(dev.index)
test = rest.groupby("policy_area").head(25)
pool = rest.drop(test.index)
seed_set = pool.groupby("policy_area").sample(10, random_state=SEED)
more_real = pool.drop(seed_set.index).groupby("policy_area").sample(200, random_state=SEED)
dev, test, seed_set, more_real = (d.reset_index(drop=True) for d in (dev, test, seed_set, more_real))
print(f"seed set: {len(seed_set)}   more real: {len(more_real)}   development: {len(dev)}   test: {len(test)}")
seed_set.groupby("policy_area").head(2)

---
## 3. Baselines: few labels and many

A TF-IDF classifier (Day 1), quick to train, so we can compare many training sets. Two baselines mark
the range: the 40 seed titles, and the seed titles plus 800 more real ones.

In [ ]:
def train_and_test(texts, labels, name, eval_df=test):
    clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                        LogisticRegression(max_iter=2000, C=10, class_weight="balanced"))
    clf.fit(list(texts), list(labels))
    pred = clf.predict(eval_df["title"])
    row = {"training set": name, "training texts": len(texts),
           "accuracy": accuracy_score(eval_df["policy_area"], pred),
           "macro-F1": f1_score(eval_df["policy_area"], pred, labels=LABELS, average="macro", zero_division=0)}
    print(f"{name:<40} {len(texts):>5} texts   accuracy {row['accuracy']:.2f}   macro-F1 {row['macro-F1']:.2f}")
    return row, clf


results = []
row, _ = train_and_test(seed_set.title, seed_set.policy_area, "40 real titles")
results.append(row)
both_real = pd.concat([seed_set, more_real])
row, judge = train_and_test(both_real.title, both_real.policy_area, "840 real titles")
results.append(row)

The gap between the two is what synthetic data would have to close. We keep the classifier trained on
840 real titles (`judge`) for Section 8, where it checks whether synthetic titles look like their
category.

---
## 4. The generator

Qwen2.5 1.5B Instruct, as on Day 4, but now we **want** variety: sampling with temperature 0.9 instead
of 0. The model returns a numbered list, which `parse_list` turns into clean titles.

In [ ]:
GEN = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(GEN)
gen_model = AutoModelForCausalLM.from_pretrained(GEN, dtype="auto", device_map="auto")


def chat(messages, max_new_tokens=350, temperature=0.9):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                                 max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                                 top_p=0.95, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)


def parse_list(text, min_words=5):
    items = []
    for line in text.splitlines():
        line = re.sub(r"^\s*(\d+[.)]|[-*•])\s*", "", line).strip().strip('*"').strip()
        if len(line.split()) >= min_words and not line.endswith(":"):      # skip "Here are ten titles:"
            items.append(line)
    return items

---
## 5. Augmentation: rewording real examples

Five rewordings of every seed title. The label stays the same, so each coded title now appears in six
forms. Takes about three minutes.

In [ ]:
def paraphrase(title, n=5):
    msgs = [{"role": "system", "content": "You reword titles of US Congressional bills. Keep the meaning and "
             "the policy area; change the wording."},
            {"role": "user", "content": f"Write {n} different rewordings of this bill title, one per line, "
             f"numbered:\n{title}"}]
    return parse_list(chat(msgs))[:n]


start = time.time()
para = pd.DataFrame([{"title": p, "policy_area": row.policy_area, "source": row.title}
                     for row in seed_set.itertuples() for p in paraphrase(row.title)])
print(f"{len(para)} paraphrases in {time.time() - start:.0f} s")
para.head(6)

---
## 6. Synthetic data from scratch

Now the LLM writes new titles for each area. Two variants:

- **from the definition only**: the codebook definition of the area, nothing else;
- **with real examples**: the definition plus three randomly chosen seed titles, to show the style
  (with the instruction not to copy them).

Each call asks for 10 titles; 5 calls per area and variant give up to 50 titles per area. That takes
about five minutes.

In [ ]:
DEFINITIONS = {
    "Health": "health care and health insurance, Medicare and Medicaid, diseases and public health, drugs "
              "and medical devices, medical research",
    "Education": "schools and teachers, colleges and universities, student aid and student loans, education programs",
    "Taxation": "taxes of any kind, the Internal Revenue Code, tax credits and deductions, the IRS",
    "Armed Forces and National Security": "the military and defense, servicemembers and veterans, intelligence",
}


def synthesize(area, n=10, examples=None):
    request = (f"Write {n} realistic, varied titles of US Congressional bills in the policy area "
               f"'{area}': {DEFINITIONS[area]}. Vary the topic, the length and the wording. "
               "One title per line, numbered.")
    if examples is not None:
        request += "\n\nFor the style, here are some real titles. Don't copy them:\n" + "\n".join(f"- {e}" for e in examples)
    msgs = [{"role": "system", "content": "You write realistic titles of US Congressional bills."},
            {"role": "user", "content": request}]
    return parse_list(chat(msgs))[:n]


def generate_set(with_examples, calls_per_area=5):
    rows = []
    for area in LABELS:
        for _ in range(calls_per_area):
            ex = seed_set[seed_set.policy_area == area].title.sample(3, random_state=int(rng.integers(1e6))).tolist() \
                if with_examples else None
            rows += [{"title": t, "policy_area": area} for t in synthesize(area, examples=ex)]
    df = pd.DataFrame(rows).drop_duplicates("title")
    return df[~df.title.isin(seed_set.title)].reset_index(drop=True)      # no copies of the seed titles


start = time.time()
synth_def = generate_set(with_examples=False)
synth_ex = generate_set(with_examples=True)
print(f"{len(synth_def)} + {len(synth_ex)} synthetic titles in {time.time() - start:.0f} s")
assert len(synth_def) and len(synth_ex), "No titles came back: print chat(...) for one prompt to see why"
synth_ex.groupby("policy_area").head(2)

> **✏️ Exercise 1**
>
> Put five synthetic titles (with examples) next to five real titles from `more_real` for each area.
> What gives the synthetic ones away? Think of length, wording, how specific they are, and the
> formulas real bills use ("To amend title 38, United States Code, ...").

In [ ]:
# Your code here

---
## 7. Did it help?

Every training set is tested on the same 100 real, human-coded test titles.

In [ ]:
sets = {
    "40 real + paraphrases": pd.concat([seed_set, para[["title", "policy_area"]]]),
    "synthetic, from definitions": synth_def,
    "synthetic, with examples": synth_ex,
    "40 real + synthetic (with examples)": pd.concat([seed_set, synth_ex]),
}
for name, df in sets.items():
    row, _ = train_and_test(df.title, df.policy_area, name)
    results.append(row)

summary = pd.DataFrame(results).set_index("training set")
ax = summary["macro-F1"].plot(kind="barh", color="#34B233", figsize=(8, 3.5))
ax.set_xlabel("Macro-F1 on 100 real test titles")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
summary.round(3)

How to read this:

- With 100 test titles, one title is one percentage point. Differences of a few points are within
  the noise of a single run.
- Li et al. (2023) find that synthetic data works best when the task is **objective** and the
  categories are easy to describe, and much less for subjective tasks such as detecting humor or
  sarcasm. Generating with real examples beat generating from instructions alone, and models
  trained on synthetic data generally did worse than models trained on real data.
- Across ten social science classification tasks, Møller et al. (2024) find that human-coded data
  did as well as or better than adding synthetic data. Synthetic data helped mostly for **rare
  classes**.
- The **840 real titles** line is the benchmark that matters: if coding more real data is possible,
  it is usually the better investment.

> **✏️ Exercise 2**
>
> **A rare category.** Suppose you had only **3** coded Education titles (and still 10 for the other
> areas). Train the classifier on that unbalanced seed set, then add the 50 synthetic Education titles
> (with examples) and train again. Compare the F1 score **for Education** on the test set.

In [ ]:
# Your code here

---
## 8. Is the synthetic data any good?

Four checks that work without coding anything new:

- **Diversity**: how different are the texts from each other? (average similarity between pairs;
  lower is more varied, using TF-IDF vectors)
- **Realism**: can a classifier tell synthetic from real titles? An AUC of 0.5 means
  indistinguishable; close to 1 means easy to spot.
- **Label quality**: does the classifier trained on 840 real titles (`judge`) put each synthetic
  title in the category it was written for?
- **Leakage**: are any synthetic titles near-copies of **test** titles? These bills are public, and
  the LLM may have seen them during training. A near-copy in the training data inflates test scores.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(bills.title)


def mean_similarity(texts, k=150):
    texts = pd.Series(texts).sample(min(k, len(texts)), random_state=SEED)
    sims = cosine_similarity(vectorizer.transform(texts))
    return sims[np.triu_indices_from(sims, k=1)].mean()


def realism_auc(synthetic, k=150):
    real = more_real.title.sample(min(k, len(synthetic)), random_state=SEED)
    fake = pd.Series(synthetic).sample(len(real), random_state=SEED)
    texts, y = pd.concat([real, fake]).tolist(), [0] * len(real) + [1] * len(fake)
    clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=2000))
    p = cross_val_predict(clf, texts, y, cv=5, method="predict_proba")[:, 1]
    return roc_auc_score(y, p)


test_vectors = vectorizer.transform(test.title)
checks = {}
for name, df in {"real (840)": both_real, "paraphrases": para, "synthetic, from definitions": synth_def,
                 "synthetic, with examples": synth_ex}.items():
    max_sim_to_test = cosine_similarity(vectorizer.transform(df.title), test_vectors).max(axis=1)
    checks[name] = {"mean similarity": mean_similarity(df.title),
                    "realism AUC": np.nan if name.startswith("real") else realism_auc(df.title),
                    "judge agrees with label": (judge.predict(df.title) == df.policy_area).mean(),
                    "near-copies of test titles": int((max_sim_to_test > 0.9).sum())}
table = pd.DataFrame(checks).T
table["near-copies of test titles"] = table["near-copies of test titles"].astype(int)
table.round(3).fillna("–")

Typical patterns: synthetic titles are **more alike** than real ones, **easy to tell apart** from real
titles, and **very often in the "right" category**: they are clear-cut, prototypical examples. That is
both their strength (clean labels) and their weakness (they rarely show the borderline cases where a
classifier needs help). Any near-copy of a test title should be removed before training, and reported.

One caveat: the judge was trained on the 840 real titles, so its agreement in that row is high by
construction. Its accuracy on the real test set (Section 3) is the fair reference point.

> **✏️ Exercise 3**
>
> **Temperature and diversity.** Generate 30 Taxation titles (3 calls, with examples) at temperature 0.5
> and 30 at temperature 1.2. Compare their mean similarity and the share the judge puts in Taxation.
> Read a few from each. What is the trade-off?

In [ ]:
# Your code here

> **✏️ Exercise 4**
>
> **How much synthetic data?** Add 10, 25 and all 50 synthetic titles per area (with examples) to the 40
> real titles, and plot macro-F1 on the **development** set against the number of synthetic titles.

In [ ]:
# Your code here

---
## 9. Risks

- **The model's view becomes your data.** Synthetic examples reflect what the LLM thinks a category
  looks like, including its stereotypes (Day 4, ethics). A classifier trained on them inherits that
  view.
- **Too clean, too similar.** Prototypical examples teach the easy cases. The hard, borderline cases,
  where measurement errors happen, are underrepresented.
- **Leakage and contamination.** Public data may be in the LLM's training data; synthetic "new" texts can
  be near-copies of real ones, including your test set.
- **Model collapse.** Models trained over generations on their own synthetic output lose the rare and
  unusual parts of the distribution (Shumailov et al., 2024). One round of augmentation is not that, but
  it points in the same direction.
- **Privacy is not automatic.** Synthetic data generated from confidential texts can repeat details from
  them. Check before sharing.
- **Report it.** Say which texts were synthetic, how they were generated (model, prompt, temperature,
  filtering), and evaluate only on real, human-coded data.

> **✏️ Exercise 5**
>
> **Memorized or new?** For every synthetic title (with examples), find the most similar title in the
> **whole** bills dataset (`bills`), and look at the ten closest matches. Are these copies of real bills?
> What would that mean for a study that uses the synthetic data?

In [ ]:
# Your code here

---
## 10. Wrap-up

- **Synthetic data trains, real data tests.** Never evaluate on synthetic texts.
- **Start from real examples.** Generating with your own coded examples beats generating from
  definitions alone, and synthetic data works best as an addition to real data, not a replacement.
- **Check it** for diversity, realism, label quality and leakage before you train on it.
- **Use it where it helps most**: rare categories, early prototypes, test cases. When you can code more
  real data, that is usually the better investment.
- **Report** what was generated, how, and how much, and keep it separate from your real data.

### Further reading

- Argyle, L. P., et al. (2023). Out of one, many: Using language models to simulate human samples.
  *Political Analysis, 31*(3).
- Bisbee, J., Clinton, J. D., Dorff, C., Kenkel, B., & Larson, J. M. (2024). Synthetic replacements for
  human survey data? The perils of large language models. *Political Analysis, 32*(4), 401–416.
- Feng, S. Y., et al. (2021). A survey of data augmentation approaches for NLP. *Findings of ACL 2021*.
- Li, Z., Zhu, H., Lu, Z., & Yin, M. (2023). Synthetic data generation with large language models for
  text classification: Potential and limitations. *Proceedings of EMNLP 2023*.
- Møller, A. G., Pera, A., Dalsgaard, J., & Aiello, L. M. (2024). The parrot dilemma: Human-labeled vs.
  LLM-augmented data in classification tasks. *Proceedings of EACL 2024 (Volume 2: Short Papers)*,
  179–192.
- Shumailov, I., Shumaylov, Z., Zhao, Y., Papernot, N., Anderson, R., & Gal, Y. (2024). AI models collapse
  when trained on recursively generated data. *Nature, 631*, 755–759.